# Intro to PandaPower for IDP Project Optimization

In [1]:
try:
    import matplotlib as plt
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries, OutputWriter
    from pandapower.diagnostic import Diagnostic
    from pandapower.create import create_storage
    from tools.limits import bus_vm_pu_limits, line_loading_limits
    from tools.jacobian import voltage_sensitivity_matrix, vs_mse   

except: 
    %pip install pandapower
    %pip install numba
    %pip install matplotlib
    %pip install scikit-learn
    import matplotlib as plt
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries
    from tools.jacobian import voltage_sensitivity_matrix, vs_mse   

from control.battery import Battery
import profileloader as pl
import numpy as np
import os
import tempfile

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 
By default, the IEEE 30 net has a dataframe for each of the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [2]:
net = case30()
print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


In [3]:
net.poly_cost


,element,et,cp0_eur,cp1_eur_per_mw,cp2_eur_per_mw2,cq0_eur,cq1_eur_per_mvar,cq2_eur_per_mvar2
0,0,ext_grid,0.0,2.00,0.02000,0.0,0.0,0.0
1,0,gen,0.0,1.75,0.01750,0.0,0.0,0.0
2,1,gen,0.0,1.00,0.06250,0.0,0.0,0.0
3,2,gen,0.0,3.25,0.00834,0.0,0.0,0.0
4,3,gen,0.0,3.00,0.02500,0.0,0.0,0.0
5,4,gen,0.0,3.00,0.02500,0.0,0.0,0.0


In [4]:
store_el = create_storage( 
    net, 20, 
    p_mw = 20, 
    q_mvar = 0, 
    controllable=True,
    max_e_mwh = 120, 
    soc_percent=50,
    max_p_mw=60, 
    min_p_mw=-60, 
    max_q_mvar=30,
    min_q_mvar=-30)

print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - storage (1 element)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


Specific cells are adjustable based on external data

In [5]:
# for example: set the power consumption of load #1 to 2x its starting value
"""print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 2 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

# and: set line loading limit to 50%
print("Before: " + str(net.line.at[10,'max_loading_percent']))
net.line.at[10,'max_loading_percent'] = 0.5
print("After: " + str(net.line.at[10,'max_loading_percent']))"""

# and: set generator 5 to limit to 2x its starting value
print("Before: " + str(net.gen.at[1,'p_mw']))
net.gen.at[4,'p_mw'] = 3 * net.gen.iloc[4].p_mw
print("After: " + str(net.gen.at[4,'p_mw']))

# and: set generator 5 to limit to 2x its starting value
print("Before: " + str(net.gen.at[3,'p_mw']))
net.gen.at[3,'p_mw'] = 3 * net.gen.iloc[3].p_mw
print("After: " + str(net.gen.at[3,'p_mw']))

Before: 21.59
After: 111.0
Before: 19.2
After: 57.599999999999994


### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [6]:
runpp(net)
vs_mse(net)

0.00046612202293460313

You can use the result dataframes to check if power flow data is within the defined limits for each object

In [7]:
# For example, buses:
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 108.2913% loading is greater than maximum of 100.0% loading
Line #15: Result 174.128% loading is greater than maximum of 100.0% loading
Line #20: Result 156.6672% loading is greater than maximum of 100.0% loading
Line #21: Result 140.8812% loading is greater than maximum of 100.0% loading
Line #22: Result 122.2404% loading is greater than maximum of 100.0% loading
Line #28: Result 143.748% loading is greater than maximum of 100.0% loading
Line #29: Result 132.5882% loading is greater than maximum of 100.0% loading
Line #30: Result 163.9677% loading is greater than maximum of 100.0% loading
Line #31: Result 217.3782% loading is greater than maximum of 100.0% loading


For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [8]:
try: 
    runopp(net)
except: 
    diag = Diagnostic()
    result = diag.diagnose_network(net, report_style="detailed")    


Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [9]:
print(net.gen.head())

   name  bus    p_mw  vm_pu  sn_mva  min_q_mvar  max_q_mvar  scaling  slack  \
0  None    1   60.97    1.0     NaN       -20.0        60.0      1.0  False   
1  None   21   21.59    1.0     NaN       -15.0        62.5      1.0  False   
2  None   26   26.91    1.0     NaN       -15.0        48.7      1.0  False   
3  None   22   57.60    1.0     NaN       -10.0        40.0      1.0  False   
4  None   12  111.00    1.0     NaN       -15.0        44.7      1.0  False   

   in_service  slack_weight  type  controllable  max_p_mw  min_p_mw  \
0        True           0.0  None          True      80.0       0.0   
1        True           0.0  None          True      50.0       0.0   
2        True           0.0  None          True      55.0       0.0   
3        True           0.0  None          True      30.0       0.0   
4        True           0.0  None          True      40.0       0.0   

   id_q_capability_characteristic  reactive_capability_curve curve_style  
0                      

In [10]:
# proof that bus max/min values are now accounted for
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
All line loading within limits


To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [11]:
net.gen.at[4,'p_mw'] = 20
net.gen.at[4, 'controllable'] = False

In [12]:
runpp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 110.8988% loading is greater than maximum of 100.0% loading
Line #28: Result 131.8907% loading is greater than maximum of 100.0% loading
Line #29: Result 195.7788% loading is greater than maximum of 100.0% loading
Line #30: Result 137.9347% loading is greater than maximum of 100.0% loading
Line #31: Result 150.3774% loading is greater than maximum of 100.0% loading


In [13]:
runopp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
All line loading within limits


The resulting optimized components have changed based on this new constraint. 

### Analysis with Custom Data

Load in custom loads/gens from the provided Excel sheets

In [14]:
net=case30()

net.poly_cost["cp2_eur_per_mw2"] = 0        # clear out default costs
store_el = create_storage( 
    net, 5, 
    p_mw = 0, 
    q_mvar = 0, 
    controllable=True,
    max_e_mwh = 20, 
    soc_percent=60,
    max_p_mw=5, 
    min_p_mw=-5, 
    max_q_mvar=5,
    min_q_mvar=-5)


load_control = pl.json_to_net(net, "load", date="1/2/2026")
gen_control = pl.json_to_net(net, "gen", date="1/2/2026")
tariff_control = pl.json_to_net(net, "poly_cost", date="1/2/2026")
storage_control = Battery(net=net, element_index=0)

In [15]:
ow = OutputWriter(net, 
                  output_path="..\\results", 
                  output_file_type='.csv', csv_separator=",")
ow.log_variable("res_storage", "p_mw")
ow.log_variable("storage", "soc_percent")
run_timeseries(net, continue_on_divergence=True
               , max_iteration=40, verbose=True)

No time steps to calculate are specified. I'll check the datasource of the first controller for avaiable time steps
  2%|▏         | 2/96 [00:00<00:09, 10.38it/s]

timestep p_mw=0.0 soc=60.0 pred=-0.10399689874562024
timestep p_mw=-0.10399689874562024 soc=59.86316197533471 pred=-0.10421423496476556


  4%|▍         | 4/96 [00:01<00:43,  2.14it/s]

timestep p_mw=-0.10421423496476556 soc=59.72603798196002 pred=-0.10130448081568655


  5%|▌         | 5/96 [00:02<00:48,  1.89it/s]

...

 99%|█████████▉| 95/96 [01:00<00:00,  1.47it/s]

timestep p_mw=-0.09553121902592145 soc=69.56696308534136 pred=-0.09679110643274341


100%|██████████| 96/96 [01:00<00:00,  1.50it/s]

timestep p_mw=-0.09679110643274341 soc=69.4396063663509 pred=-0.10142121568362666


100%|██████████| 96/96 [01:01<00:00,  1.56it/s]


In [16]:
net.poly_cost

,element,et,cp0_eur,cp1_eur_per_mw,cp2_eur_per_mw2,cq0_eur,cq1_eur_per_mvar,cq2_eur_per_mvar2
0,0,ext_grid,0.0,0.000132,0,0.0,0.0,0.0
1,0,gen,0.0,0.000000,0,0.0,0.0,0.0
2,1,gen,0.0,0.000000,0,0.0,0.0,0.0
3,2,gen,0.0,0.000000,0,0.0,0.0,0.0
4,3,gen,0.0,0.000000,0,0.0,0.0,0.0
5,4,gen,0.0,0.000000,0,0.0,0.0,0.0
